In [1]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

In [81]:
import os
import sys
from pathlib import Path
sys.path.append('/lab/barcheese01/smaffa/coTISja/src')

from scripts.filter_utils import *
import re

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

deseq2 = importr("DESeq2")
apeglm = importr("apeglm")

In [3]:
experiment_table, sample_df, replicate_df = load_experiment_manifest()
tisdiff_manifest = pd.read_csv('/lab/barcheese01/smaffa/coTISja/data/ribotish_tisdiff_manifest.tsv', sep='\t')
samples = experiment_table['sample'].tolist()
codon_order = ['ATG', 'ATA', 'ATC', 'ATT', 'ACG', 'AAG', 'AGG', 'GTG', 'TTG', 'CTG']

In [4]:
pairs_to_count_matrix = dict()
for _, r in tisdiff_manifest.iterrows():
    baseline_s = r['baseline']
    test_s = r['test']
    counts_file = r['export_output_file']

    counts_matrix = pd.read_csv(counts_file, sep='\t', index_col=0)

    sample_pair  = [baseline_s, test_s]
    n_reps = counts_matrix.shape[1] // 4
    assays = ['rpf', 'rna']
    
    column_names = []
    for a in assays:
        for s in sample_pair:
            for i in range(1, n_reps+1):
                column_names.append(f'{s}__rep{i}__{a}')
    counts_matrix.columns = column_names
    pairs_to_count_matrix[(test_s, baseline_s)] = counts_matrix

In [8]:
# need to backfill RNA counts by GENE (for TISs which may only be absent in some samples), which requires a TIS ID to GENE ID mapping
# can get this mapping from a global concatenation of predict files
global_mapping = None
for pf in replicate_df[replicate_df['condition'] == 'TIS']['predict_file'].tolist():
    id_map = pd.read_csv(pf, sep='\t')[['Gid', 'Tid']].drop_duplicates()
    if global_mapping is None:
        global_mapping = id_map
    else:
        global_mapping = pd.concat([global_mapping, id_map], axis=0).drop_duplicates()
global_mapping = global_mapping.reset_index(drop=True)

In [ ]:
def generate_coldata_table(column_names, sep='__'):
    metadata_df = pd.DataFrame({
        'colname': column_names,
        'condition': [x.split(sep)[0] for x in column_names],
        'replicate': [x.split(sep)[1] for x in column_names],
        'assay': [x.split(sep)[2] for x in column_names]
    }).set_index('colname')
    return metadata_df

def assign_tids_to_tis_matrix(tis_matrix):
    matrix_copy = tis_matrix.copy()
    matrix_copy['Tid'] = tis_matrix.index.str.split('_').str[0]
    return matrix_copy.reset_index()

In [82]:
# a bit convoluted because ribotish has already merged per-TIS Riboseq (rpf) counts with per-gene RNAseq (rna) counts, but we need to re-extract TIS rpf counts and gene rna counts
# 1) extract this rpf per TIS and rna per gene
# 2) concatenate all rpf counts (with TIS index), then map the TIS ids to corresponding transcript and gene ids
# 3) concatenate all RNA counts (with gene index)
# 4) merge the rpf matrix with the rna matrix

rpf_counts_by_sample = dict() 
rna_counts_by_replicate = dict()
for pair, counts in pairs_to_count_matrix.items():
    s1, s2 = pair

    if s1 not in rpf_counts_by_sample:
        all_sample_columns = [c for c in counts.columns if s1 in c]
        rpf_columns = [c for c in all_sample_columns if 'rpf' in c]
        rna_columns = [c for c in all_sample_columns if 'rna' in c]

        rpf_counts_by_sample[s1] = counts.loc[:, rpf_columns] # for rpf we concatenate first before mapping
        replicate_rna_counts = assign_tids_to_tis_matrix(counts.loc[:, rna_columns]).merge(global_mapping, how='left')  # for rna we map to transcript and gene ids
        for rep in rna_columns:
            rna_counts_by_replicate[rep] = replicate_rna_counts.loc[:, ['Gid', rep]].drop_duplicates().set_index('Gid')[rep] # get non-duplicated entries per gene
    if s2 not in rpf_counts_by_sample:
        all_sample_columns = [c for c in counts.columns if s2 in c]
        rpf_columns = [c for c in all_sample_columns if 'rpf' in c]
        rna_columns = [c for c in all_sample_columns if 'rna' in c]

        rpf_counts_by_sample[s2] = counts.loc[:, rpf_columns]
        replicate_rna_counts = assign_tids_to_tis_matrix(counts.loc[:, rna_columns]).merge(global_mapping, how='left')
        for rep in rna_columns:
            rna_counts_by_replicate[rep] = replicate_rna_counts.loc[:, ['Gid', rep]].drop_duplicates().set_index('Gid')[rep]

# merge the rpf counts and rna counts independently
all_rpf_counts = assign_tids_to_tis_matrix(pd.concat(rpf_counts_by_sample.values(), axis=1)).merge(global_mapping, how='left').fillna(0)
all_rna_counts = pd.concat(rna_counts_by_replicate, axis=1).fillna(0)
# map the counts together on gene id
all_counts = all_rpf_counts.merge(all_rna_counts, how='left', left_on='Gid', right_index=True).drop(['Tid', 'Gid'], axis=1).set_index('TIS')
all_counts = all_counts.astype(int)
# generate a sample metadata file for DESeq2
all_coldata = generate_coldata_table(all_counts.columns)

# Run DESeq2

From [Bioconductor forums](https://support.bioconductor.org/p/61509/):

    DESeq2 testing ratio of ratios (RIP-Seq, CLIP-Seq, ribosomal profiling)

    Suppose we have two assays: Input and IP, and we have two conditions: A and B.

    Then using a design: '~ assay + condition + assay:condition', the interaction term 'assay:condition' represents the ratio of the ratios: (IP for B / Input for B) / (IP for A / Input for A)... which with some algebra you can see is equivalent to (IP for B / IP for A) / (Input for B / Input for A).

    Using a likelihood ratio test which removes the interaction term in the reduced model, we test whether the IP enrichment is different in condition B than in A:

    dds <- DESeqDataSet(se, design= ~ assay + condition + assay:condition)
    dds <- DESeq(dds, test="LRT", reduced= ~ assay + condition)
    results(dds)

~Michael Love, lab of authors of DESeq2

In this sense, the LRT (likelihood ratio test) hypothesis test is whether there is ANY difference in translation efficiency (TE) between conditions, performed by comparing a model with and without the ratio term between riboseq and RNA

With exactly two conditions, this is roughly equivalent, but non-directional, to the Wald Test comparing the magnitude of the interaction coefficient to 0 (i.e. no difference in TE). But with more conditions, this distinction becomes clearer.

Specifically, the Wald Test will give, for each pair of condtions, the magnitude of the difference in translational efficiency; whereas the LRT still tests whether there is ANY difference in translation efficiency between any conditions.

Both are perhaps valuable: Wald giving pairwise foldchanges; and LRT giving a conservative measure of any outlier in TE.

## Prototype running DESeq through rpy2

In [ ]:
# counts = pairs_to_count_matrix[('K562', 'HeLa')]
# coldata = generate_coldata_table(counts.columns)

# with localconverter(ro.default_converter + pandas2ri.converter):
#     r_counts = pandas2ri.py2rpy(counts)
#     r_coldata = pandas2ri.py2rpy(coldata)

# r_dds = deseq2.DESeqDataSetFromMatrix(countData=r_counts, colData=r_coldata, design=ro.Formula('~ assay + condition + assay:condition'))
# r_dds = deseq2.DESeq(r_dds, test='LRT', reduced=ro.Formula('~ assay + condition'))

# r_res_df = ro.r("as.data.frame")(deseq2.results(r_dds))

# with localconverter(ro.default_converter + pandas2ri.converter):
#     res_df = ro.conversion.rpy2py(r_res_df)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: -- note: fitType='parametric', but the dispersion trend was not well captured by the
   function: y = a/x + b, and a local regression fit was automatically substituted.
   specify fitType='local' or 'mean' to avoid this message next time.

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



## Two questions can be answered with DESeq2

1) For any arbitrary pair of cell lines to compare, what is the difference in TE usage across cell lines/conditions?
    - Fit the global RNA counts and RPF counts matrix with 3 terms: the RNA counts, the RPF counts, and the interaction between the RNA and RPF counts: `~ assay + cell_line + assay:cell_line`
    - The coefficients of the interaction term should capture the differences in translational efficiency between cell lines
    - All pairwise LFCs (in translational efficiency) can be extracted with the `contrast` parameter for the interaction terms
    - A conservative measure of whether TE usage differs in any cell line can be tested with the `LRT` test

2) For a correlative analysis across all cell lines, how do the fitted means vary across cell lines?
    - Fit the global RNA counts and RPF counts matrix with 3 terms again
    - The means per sample can be extracted and compared across cell lines
    - A variance-stabilizing transformation (`vst()`) may be helpful so make low-count sites comparable to high-count sites, which according to the statistical model of NB-distributed counts, higher average counts necessarily lead to higher variances (mean-variance relationship; exacerbated by overdispersion)

## Global DESeq2 fit over all conditions

In [85]:
# firstly, export the counts matrix and the sample metadata table
condition_counts = all_counts.T.merge(all_coldata[['condition', 'assay']], left_index=True, right_index=True).groupby(['condition', 'assay']).sum().T.loc[:, pd.IndexSlice[:, 'rpf']].droplevel(1, axis=1)
tis_mask = condition_counts >= 5
condition_rna_counts = all_counts.T.merge(all_coldata[['condition', 'assay']], left_index=True, right_index=True).groupby(['condition', 'assay']).sum().T.loc[:, pd.IndexSlice[:, 'rna']].droplevel(1, axis=1)
rna_mask = condition_rna_counts >= 5

In [ ]:
condition_counts.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/global/rpf_summed_replicate_counts.csv')
condition_counts[tis_mask].to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/global/rpf_summed_replicate_counts_masked.csv')
condition_rna_counts.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/global/rna_summed_replicate_counts.csv')
condition_rna_counts[rna_mask].to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/global/rna_summed_replicate_counts_masked.csv')

In [ ]:
all_counts.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/global/input_counts_matrix.csv')
all_coldata.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/global/input_sample_metadata.csv')

### Run the LRT test for any difference in TE

In [83]:
with localconverter(ro.default_converter + pandas2ri.converter):
    r_counts = pandas2ri.py2rpy(all_counts)
    r_coldata = pandas2ri.py2rpy(all_coldata)

r_dds = deseq2.DESeqDataSetFromMatrix(countData=r_counts, colData=r_coldata, design=ro.Formula('~ assay + condition + assay:condition'))
r_lrt_dds = deseq2.DESeq(r_dds, test='LRT', reduced=ro.Formula('~ assay + condition'))
r_lrt_res_df = ro.r("as.data.frame")(deseq2.results(r_lrt_dds))

with localconverter(ro.default_converter + pandas2ri.converter):
    lrt_res_df = ro.conversion.rpy2py(r_lrt_res_df)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [84]:
lrt_res_df.sort_values('padj').to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/global/lrt_results.csv')

## Run all pairwise Wald tests for relative LFCs

In [87]:
with localconverter(ro.default_converter + pandas2ri.converter):
    r_counts = pandas2ri.py2rpy(all_counts)
    r_coldata = pandas2ri.py2rpy(all_coldata)

r_dds = deseq2.DESeqDataSetFromMatrix(countData=r_counts, colData=r_coldata, design=ro.Formula('~ assay + condition + assay:condition'))
r_wald_dds = deseq2.DESeq(r_dds)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [88]:
# determine which sample was used as the reference by DESeq2 (simply alphabetical)
ro.globalenv["r_coldata"] = r_coldata
ro.globalenv["r_wald_dds"] = r_wald_dds
reference_sample = ro.r('levels(factor(r_coldata$condition))')[0]

In [89]:
# for all pairwise comparisons, extract the results
pairwise_wald_dict = dict()

# conversion can't recognize string operators (-) well --> need to instead run this in r eval strings
for pair in tqdm(pairs_to_count_matrix):
    s1, s2 = pair
    if s1 == reference_sample:
        # swap the variables
        temp = s1
        s1 = s2
        s2 = temp
    if s2 == reference_sample:
        coeff_name = f'assayrpf.condition{s1}'
        res = ro.r(f'as.data.frame(results(r_wald_dds, name="{coeff_name}"))')
    else:
        contrast = f'list("assayrpf.condition{s1}", "assayrpf.condition{s2}")'
        res = ro.r(f'as.data.frame(results(r_wald_dds, contrast={contrast}, listValues=c(1, -1)))')
    
    with localconverter(ro.default_converter + pandas2ri.converter):
        pair_res_df = pandas2ri.rpy2py(res)

    pairwise_wald_dict[(s1, s2)] = pair_res_df

100%|██████████| 15/15 [02:20<00:00,  9.39s/it]


In [93]:
for p, df in tqdm(pairwise_wald_dict.items()):
    s1, s2 = p
    df.to_csv(f'/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/global/pairwise_wald/{s1}_vs_{s2}.csv')

100%|██████████| 15/15 [00:47<00:00,  3.18s/it]


## Run pairwise DESeq models to confirm transcriptional differences between conditions

In [94]:
rna_counts = assign_tids_to_tis_matrix(all_counts[[c for c in all_counts.columns if 'rna' in c]]).merge(global_mapping, how='left')
rna_counts = rna_counts.drop_duplicates(subset=['Gid']).set_index('Gid').drop(['TIS', 'Tid'], axis=1)
rna_coldata = all_coldata[all_coldata['assay'] == 'rna']

In [96]:
with localconverter(ro.default_converter + pandas2ri.converter):
    r_rna_counts = pandas2ri.py2rpy(rna_counts)
    r_rna_coldata = pandas2ri.py2rpy(rna_coldata)

r_rna_dds = deseq2.DESeqDataSetFromMatrix(countData=r_rna_counts, colData=r_rna_coldata, design=ro.Formula('~ condition'))
r_rna_wald_dds = deseq2.DESeq(r_rna_dds)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: -- note: fitType='parametric', but the dispersion trend was not well captured by the
   function: y = a/x + b, and a local regression fit was automatically substituted.
   specify fitType='local' or 'mean' to avoid this message next time.

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [97]:
# determine which sample was used as the reference by DESeq2 (simply alphabetical)
ro.globalenv["r_rna_coldata"] = r_rna_coldata
ro.globalenv["r_rna_wald_dds"] = r_rna_wald_dds
reference_sample = ro.r('levels(factor(r_rna_coldata$condition))')[0]

In [98]:
# for all pairwise comparisons, extract the results
pairwise_wald_rna_dict = dict()

# conversion can't recognize string operators (-) well --> need to instead run this in r eval strings
for pair in tqdm(pairs_to_count_matrix):
    s1, s2 = pair
    if s1 == reference_sample:
        # swap the variables
        temp = s1
        s1 = s2
        s2 = temp
    if s2 == reference_sample:
        coeff_name = f'condition_{s1}_vs_{reference_sample}'
        res = ro.r(f'as.data.frame(results(r_rna_wald_dds, name="{coeff_name}"))')
    else:
        contrast = f'list("condition_{s1}_vs_{reference_sample}", "condition_{s2}_vs_{reference_sample}")'
        res = ro.r(f'as.data.frame(results(r_rna_wald_dds, contrast={contrast}, listValues=c(1, -1)))')
    
    with localconverter(ro.default_converter + pandas2ri.converter):
        pair_res_df = pandas2ri.rpy2py(res)

    pairwise_wald_rna_dict[(s1, s2)] = pair_res_df

100%|██████████| 15/15 [00:07<00:00,  1.93it/s]


In [99]:
for p, df in tqdm(pairwise_wald_rna_dict.items()):
    s1, s2 = p
    df.to_csv(f'/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/global/pairwise_wald_rna/{s1}_vs_{s2}.csv')

100%|██████████| 15/15 [00:04<00:00,  3.72it/s]


## Extract the fitted means for the Riboseq data and the TE data and compare them directly without VST

In [100]:
# determine which sample was used as the reference by DESeq2 (simply alphabetical)
ro.globalenv["r_coldata"] = r_coldata
ro.globalenv["r_wald_dds"] = r_wald_dds
reference_sample = ro.r('levels(factor(r_coldata$condition))')[0]
non_reference_samples = sorted(list(set(samples) - {reference_sample}))

In [101]:
te_coefficients = []
with localconverter(ro.default_converter + pandas2ri.converter):
    baseline = ro.conversion.rpy2py(ro.r('coef(r_wald_dds)[, "assay_rpf_vs_rna"]'))
    te_coefficients.append(baseline)
    for s in non_reference_samples:
        addend_coef = ro.conversion.rpy2py(ro.r(f'coef(r_wald_dds)[, "assayrpf.condition{s}"]'))
        te_coefficients.append(baseline + addend_coef)
log_te = pd.DataFrame(te_coefficients, index=[reference_sample] + non_reference_samples, columns=all_counts.index).T


In [105]:
log_te.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/global/log_translation_efficiency_coeffs.csv')

## Extract the variance-stabilized transform matrix of mean TE per condition

In [106]:
# run the transformation
r_vst = deseq2.vst(r_dds, blind=True)

# extract the transformed matrix
r_vst_mtx = ro.r('assay')(r_vst)
with localconverter(ro.default_converter + pandas2ri.converter):
    vst_mtx = pd.DataFrame(ro.conversion.rpy2py(r_vst_mtx), index=all_counts.index, columns=all_counts.columns)

# both RNA and Riboseq reads have been variance stabilized and logged (i.e. adjusted for mean-variance relationship), average over replicates per sample
vst_avg_mtx = vst_mtx.T.groupby([all_coldata['condition'], all_coldata['assay']]).mean().T

# subtracting log(RPF) - log(RNA) gives log(RPF/RNA) ~= log(TE)
vst_avg_te = vst_avg_mtx.loc[:, pd.IndexSlice[:, 'rpf']].droplevel(1, axis=1) - vst_avg_mtx.loc[:, pd.IndexSlice[:, 'rna']].droplevel(1, axis=1)

vst_avg_te.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/global/translation_efficiency_vst_matrix.csv')

# Repeat DESeq2 model fitting & analysis, but on the relevant sample "blocks" for comparison

## Across cell lines

In [181]:
subset_sample_names = ['HeLa', 'K562', 'RPE1_Async', 'U2OS']
subset_coldata = all_coldata[all_coldata['condition'].isin(subset_sample_names)]
subset_counts = all_counts.loc[:, subset_coldata.index.tolist()]
subset_counts = subset_counts[(subset_counts[[c for c in subset_counts.columns if 'rpf' in c]].sum(axis=1) > 0)]

In [182]:
subset_counts.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/cell_lines/input_counts_matrix.csv')
subset_coldata.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/cell_lines/input_sample_metadata.csv')

### LRT

In [183]:
with localconverter(ro.default_converter + pandas2ri.converter):
    r_counts = pandas2ri.py2rpy(subset_counts)
    r_coldata = pandas2ri.py2rpy(subset_coldata)

r_dds = deseq2.DESeqDataSetFromMatrix(countData=r_counts, colData=r_coldata, design=ro.Formula('~ assay + condition + assay:condition'))
r_lrt_dds = deseq2.DESeq(r_dds, test='LRT', reduced=ro.Formula('~ assay + condition'))
r_lrt_res_df = ro.r("as.data.frame")(deseq2.results(r_lrt_dds))

with localconverter(ro.default_converter + pandas2ri.converter):
    lrt_res_df = ro.conversion.rpy2py(r_lrt_res_df)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [184]:
lrt_res_df.sort_values('padj').to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/cell_lines/lrt_results.csv')

### Pairwise TIS/RNA

In [185]:
with localconverter(ro.default_converter + pandas2ri.converter):
    r_counts = pandas2ri.py2rpy(subset_counts)
    r_coldata = pandas2ri.py2rpy(subset_coldata)

r_dds = deseq2.DESeqDataSetFromMatrix(countData=r_counts, colData=r_coldata, design=ro.Formula('~ assay + condition + assay:condition'))
r_wald_dds = deseq2.DESeq(r_dds)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [186]:
# determine which sample was used as the reference by DESeq2 (simply alphabetical)
ro.globalenv["r_coldata"] = r_coldata
ro.globalenv["r_wald_dds"] = r_wald_dds
reference_sample = ro.r('levels(factor(r_coldata$condition))')[0]

In [187]:
# for all pairwise comparisons, extract the results
pairwise_wald_dict = dict()

# conversion can't recognize string operators (-) well --> need to instead run this in r eval strings
for pair in tqdm(pairs_to_count_matrix):
    s1, s2 = pair
    if s1 in subset_sample_names and s2 in subset_sample_names:
        if s1 == reference_sample:
            # swap the variables
            temp = s1
            s1 = s2
            s2 = temp
        if s2 == reference_sample:
            coeff_name = f'assayrpf.condition{s1}'
            res = ro.r(f'as.data.frame(results(r_wald_dds, name="{coeff_name}"))')
        else:
            contrast = f'list("assayrpf.condition{s1}", "assayrpf.condition{s2}")'
            res = ro.r(f'as.data.frame(results(r_wald_dds, contrast={contrast}, listValues=c(1, -1)))')
        
        with localconverter(ro.default_converter + pandas2ri.converter):
            pair_res_df = pandas2ri.rpy2py(res)

        pairwise_wald_dict[(s1, s2)] = pair_res_df

100%|██████████| 15/15 [00:18<00:00,  1.24s/it]


In [188]:
for p, df in tqdm(pairwise_wald_dict.items()):
    s1, s2 = p
    df.to_csv(f'/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/cell_lines/pairwise_wald/{s1}_vs_{s2}.csv')

100%|██████████| 6/6 [00:19<00:00,  3.21s/it]


### Pairwise RNA

In [189]:
rna_counts = assign_tids_to_tis_matrix(subset_counts[[c for c in subset_counts.columns if 'rna' in c]]).merge(global_mapping, how='left')
rna_counts = rna_counts.drop_duplicates(subset=['Gid']).set_index('Gid').drop(['TIS', 'Tid'], axis=1)
rna_coldata = subset_coldata[subset_coldata['assay'] == 'rna']

In [190]:
with localconverter(ro.default_converter + pandas2ri.converter):
    r_rna_counts = pandas2ri.py2rpy(rna_counts)
    r_rna_coldata = pandas2ri.py2rpy(rna_coldata)

r_rna_dds = deseq2.DESeqDataSetFromMatrix(countData=r_rna_counts, colData=r_rna_coldata, design=ro.Formula('~ condition'))
r_rna_wald_dds = deseq2.DESeq(r_rna_dds)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [191]:
# determine which sample was used as the reference by DESeq2 (simply alphabetical)
ro.globalenv["r_rna_coldata"] = r_rna_coldata
ro.globalenv["r_rna_wald_dds"] = r_rna_wald_dds
reference_sample = ro.r('levels(factor(r_rna_coldata$condition))')[0]

In [192]:
# for all pairwise comparisons, extract the results
pairwise_wald_rna_dict = dict()

# conversion can't recognize string operators (-) well --> need to instead run this in r eval strings
for pair in tqdm(pairs_to_count_matrix):
    s1, s2 = pair
    if s1 in subset_sample_names and s2 in subset_sample_names:
        if s1 == reference_sample:
            # swap the variables
            temp = s1
            s1 = s2
            s2 = temp
        if s2 == reference_sample:
            coeff_name = f'condition_{s1}_vs_{reference_sample}'
            res = ro.r(f'as.data.frame(results(r_rna_wald_dds, name="{coeff_name}"))')
        else:
            contrast = f'list("condition_{s1}_vs_{reference_sample}", "condition_{s2}_vs_{reference_sample}")'
            res = ro.r(f'as.data.frame(results(r_rna_wald_dds, contrast={contrast}, listValues=c(1, -1)))')
        
        with localconverter(ro.default_converter + pandas2ri.converter):
            pair_res_df = pandas2ri.rpy2py(res)

        pairwise_wald_rna_dict[(s1, s2)] = pair_res_df

100%|██████████| 15/15 [00:02<00:00,  5.76it/s]


In [193]:
for p, df in tqdm(pairwise_wald_rna_dict.items()):
    s1, s2 = p
    df.to_csv(f'/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/cell_lines/pairwise_wald_rna/{s1}_vs_{s2}.csv')

  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:01<00:00,  3.74it/s]


### Fitted means

In [194]:
# determine which sample was used as the reference by DESeq2 (simply alphabetical)
ro.globalenv["r_coldata"] = r_coldata
ro.globalenv["r_wald_dds"] = r_wald_dds
reference_sample = ro.r('levels(factor(r_coldata$condition))')[0]
non_reference_samples = sorted(list(set(subset_sample_names) - {reference_sample}))

In [195]:
te_coefficients = []
with localconverter(ro.default_converter + pandas2ri.converter):
    baseline = ro.conversion.rpy2py(ro.r('coef(r_wald_dds)[, "assay_rpf_vs_rna"]'))
    te_coefficients.append(baseline)
    for s in non_reference_samples:
        addend_coef = ro.conversion.rpy2py(ro.r(f'coef(r_wald_dds)[, "assayrpf.condition{s}"]'))
        te_coefficients.append(baseline + addend_coef)
log_te = pd.DataFrame(te_coefficients, index=[reference_sample] + non_reference_samples, columns=subset_counts.index).T


In [196]:
log_te.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/cell_lines/log_translation_efficiency_coeffs.csv')

### Variance-stabilized means

In [197]:
# run the transformation
r_vst = deseq2.vst(r_dds, blind=True)

# extract the transformed matrix
r_vst_mtx = ro.r('assay')(r_vst)
with localconverter(ro.default_converter + pandas2ri.converter):
    vst_mtx = pd.DataFrame(ro.conversion.rpy2py(r_vst_mtx), index=subset_counts.index, columns=subset_counts.columns)

# both RNA and Riboseq reads have been variance stabilized and logged (i.e. adjusted for mean-variance relationship), average over replicates per sample
vst_avg_mtx = vst_mtx.T.groupby([subset_coldata['condition'], subset_coldata['assay']]).mean().T

# subtracting log(RPF) - log(RNA) gives log(RPF/RNA) ~= log(TE)
vst_avg_te = vst_avg_mtx.loc[:, pd.IndexSlice[:, 'rpf']].droplevel(1, axis=1) - vst_avg_mtx.loc[:, pd.IndexSlice[:, 'rna']].droplevel(1, axis=1)

vst_avg_te.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/cell_lines/translation_efficiency_vst_matrix.csv')

## Across cell states for RPE1

In [161]:
subset_sample_names = ['RPE1_Async', 'RPE1_Que', 'RPE1_Sen']
subset_coldata = all_coldata[all_coldata['condition'].isin(subset_sample_names)]
subset_counts = all_counts.loc[:, subset_coldata.index.tolist()]
subset_counts = subset_counts[(subset_counts[[c for c in subset_counts.columns if 'rpf' in c]].sum(axis=1) > 0)]

In [163]:
subset_counts.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/RPE1_states/input_counts_matrix.csv')
subset_coldata.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/RPE1_states/input_sample_metadata.csv')

### LRT

In [164]:
with localconverter(ro.default_converter + pandas2ri.converter):
    r_counts = pandas2ri.py2rpy(subset_counts)
    r_coldata = pandas2ri.py2rpy(subset_coldata)

r_dds = deseq2.DESeqDataSetFromMatrix(countData=r_counts, colData=r_coldata, design=ro.Formula('~ assay + condition + assay:condition'))
r_lrt_dds = deseq2.DESeq(r_dds, test='LRT', reduced=ro.Formula('~ assay + condition'))
r_lrt_res_df = ro.r("as.data.frame")(deseq2.results(r_lrt_dds))

with localconverter(ro.default_converter + pandas2ri.converter):
    lrt_res_df = ro.conversion.rpy2py(r_lrt_res_df)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: -- note: fitType='parametric', but the dispersion trend was not well captured by the
   function: y = a/x + b, and a local regression fit was automatically substituted.
   specify fitType='local' or 'mean' to avoid this message next time.

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [165]:
lrt_res_df.sort_values('padj').to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/RPE1_states/lrt_results.csv')

### Pairwise TIS/RNA

In [166]:
with localconverter(ro.default_converter + pandas2ri.converter):
    r_counts = pandas2ri.py2rpy(subset_counts)
    r_coldata = pandas2ri.py2rpy(subset_coldata)

r_dds = deseq2.DESeqDataSetFromMatrix(countData=r_counts, colData=r_coldata, design=ro.Formula('~ assay + condition + assay:condition'))
r_wald_dds = deseq2.DESeq(r_dds)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: -- note: fitType='parametric', but the dispersion trend was not well captured by the
   function: y = a/x + b, and a local regression fit was automatically substituted.
   specify fitType='local' or 'mean' to avoid this message next time.

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [167]:
# determine which sample was used as the reference by DESeq2 (simply alphabetical)
ro.globalenv["r_coldata"] = r_coldata
ro.globalenv["r_wald_dds"] = r_wald_dds
reference_sample = ro.r('levels(factor(r_coldata$condition))')[0]

In [168]:
# for all pairwise comparisons, extract the results
pairwise_wald_dict = dict()

# conversion can't recognize string operators (-) well --> need to instead run this in r eval strings
for pair in tqdm(pairs_to_count_matrix):
    s1, s2 = pair
    if s1 in subset_sample_names and s2 in subset_sample_names:
        if s1 == reference_sample:
            # swap the variables
            temp = s1
            s1 = s2
            s2 = temp
        if s2 == reference_sample:
            coeff_name = f'assayrpf.condition{s1}'
            res = ro.r(f'as.data.frame(results(r_wald_dds, name="{coeff_name}"))')
        else:
            contrast = f'list("assayrpf.condition{s1}", "assayrpf.condition{s2}")'
            res = ro.r(f'as.data.frame(results(r_wald_dds, contrast={contrast}, listValues=c(1, -1)))')
        
        with localconverter(ro.default_converter + pandas2ri.converter):
            pair_res_df = pandas2ri.rpy2py(res)

        pairwise_wald_dict[(s1, s2)] = pair_res_df

100%|██████████| 15/15 [00:03<00:00,  4.40it/s]


In [169]:
for p, df in tqdm(pairwise_wald_dict.items()):
    s1, s2 = p
    df.to_csv(f'/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/RPE1_states/pairwise_wald/{s1}_vs_{s2}.csv')

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:04<00:00,  1.52s/it]


### Pairwise RNA

In [170]:
rna_counts = assign_tids_to_tis_matrix(subset_counts[[c for c in subset_counts.columns if 'rna' in c]]).merge(global_mapping, how='left')
rna_counts = rna_counts.drop_duplicates(subset=['Gid']).set_index('Gid').drop(['TIS', 'Tid'], axis=1)
rna_coldata = subset_coldata[subset_coldata['assay'] == 'rna']

In [171]:
with localconverter(ro.default_converter + pandas2ri.converter):
    r_rna_counts = pandas2ri.py2rpy(rna_counts)
    r_rna_coldata = pandas2ri.py2rpy(rna_coldata)

r_rna_dds = deseq2.DESeqDataSetFromMatrix(countData=r_rna_counts, colData=r_rna_coldata, design=ro.Formula('~ condition'))
r_rna_wald_dds = deseq2.DESeq(r_rna_dds)

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing



In [172]:
# determine which sample was used as the reference by DESeq2 (simply alphabetical)
ro.globalenv["r_rna_coldata"] = r_rna_coldata
ro.globalenv["r_rna_wald_dds"] = r_rna_wald_dds
reference_sample = ro.r('levels(factor(r_rna_coldata$condition))')[0]

In [173]:
# for all pairwise comparisons, extract the results
pairwise_wald_rna_dict = dict()

# conversion can't recognize string operators (-) well --> need to instead run this in r eval strings
for pair in tqdm(pairs_to_count_matrix):
    s1, s2 = pair
    if s1 in subset_sample_names and s2 in subset_sample_names:
        if s1 == reference_sample:
            # swap the variables
            temp = s1
            s1 = s2
            s2 = temp
        if s2 == reference_sample:
            coeff_name = f'condition_{s1}_vs_{reference_sample}'
            res = ro.r(f'as.data.frame(results(r_rna_wald_dds, name="{coeff_name}"))')
        else:
            contrast = f'list("condition_{s1}_vs_{reference_sample}", "condition_{s2}_vs_{reference_sample}")'
            res = ro.r(f'as.data.frame(results(r_rna_wald_dds, contrast={contrast}, listValues=c(1, -1)))')
        
        with localconverter(ro.default_converter + pandas2ri.converter):
            pair_res_df = pandas2ri.rpy2py(res)

        pairwise_wald_rna_dict[(s1, s2)] = pair_res_df

100%|██████████| 15/15 [00:01<00:00, 14.08it/s]


In [174]:
for p, df in tqdm(pairwise_wald_rna_dict.items()):
    s1, s2 = p
    df.to_csv(f'/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/RPE1_states/pairwise_wald_rna/{s1}_vs_{s2}.csv')

100%|██████████| 3/3 [00:00<00:00,  3.95it/s]


### Fitted means

In [175]:
# determine which sample was used as the reference by DESeq2 (simply alphabetical)
ro.globalenv["r_coldata"] = r_coldata
ro.globalenv["r_wald_dds"] = r_wald_dds
reference_sample = ro.r('levels(factor(r_coldata$condition))')[0]
non_reference_samples = sorted(list(set(subset_sample_names) - {reference_sample}))

In [ ]:
te_coefficients = []
with localconverter(ro.default_converter + pandas2ri.converter):
    baseline = ro.conversion.rpy2py(ro.r('coef(r_wald_dds)[, "assay_rpf_vs_rna"]'))
    te_coefficients.append(baseline)
    for s in non_reference_samples:
        addend_coef = ro.conversion.rpy2py(ro.r(f'coef(r_wald_dds)[, "assayrpf.condition{s}"]'))
        te_coefficients.append(baseline + addend_coef)
log_te = pd.DataFrame(te_coefficients, index=[reference_sample] + non_reference_samples, columns=subset_counts.index).T

In [177]:
log_te.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/RPE1_states/log_translation_efficiency_coeffs.csv')

### Variance-stabilized means

In [180]:
# run the transformation
r_vst = deseq2.vst(r_dds, blind=True)

# extract the transformed matrix
r_vst_mtx = ro.r('assay')(r_vst)
with localconverter(ro.default_converter + pandas2ri.converter):
    vst_mtx = pd.DataFrame(ro.conversion.rpy2py(r_vst_mtx), index=subset_counts.index, columns=subset_counts.columns)

# both RNA and Riboseq reads have been variance stabilized and logged (i.e. adjusted for mean-variance relationship), average over replicates per sample
vst_avg_mtx = vst_mtx.T.groupby([subset_coldata['condition'], subset_coldata['assay']]).mean().T

# subtracting log(RPF) - log(RNA) gives log(RPF/RNA) ~= log(TE)
vst_avg_te = vst_avg_mtx.loc[:, pd.IndexSlice[:, 'rpf']].droplevel(1, axis=1) - vst_avg_mtx.loc[:, pd.IndexSlice[:, 'rna']].droplevel(1, axis=1)

vst_avg_te.to_csv('/lab/barcheese01/smaffa/coTISja/data/tisdiff_results/deseq2/RPE1_states/translation_efficiency_vst_matrix.csv')